# 07 — Regressão Logística para Classificação

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia Técnico](./reference_materials/07_regressao_logistica.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_regressao_logistica.pdf)


## 1. Fundamentos, objetivos e contexto

**Objetivos.** Conectar GLM/logit da Disciplina 03 à classificação preditiva, explicar regularização, probabilidades, limiares e interpretação.

Apesar do nome, regressão logística é usualmente classificador. Seus coeficientes operam nos *log-odds*, não diretamente em probabilidades. `predict_proba` produz estimativas que podem exigir calibração; `predict` escolhe classes conforme regra padrão de decisão. A regularização controla complexidade.


## 2. Fundamentos matemáticos

$p(y=1\mid x)=\sigma(\beta_0+x^\top\beta)$, onde $\sigma(t)=1/(1+e^{-t})$. O logit é $\log\frac p{1-p}=\beta_0+x^\top\beta$. Perda logística binária: $-\frac1n\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]$.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score,precision_recall_fscore_support
X,y=load_breast_cancer(return_X_y=True)
a,b,c,d=train_test_split(X,y,stratify=y,test_size=.25,random_state=42)
pipe=make_pipeline(StandardScaler(),LogisticRegression(C=1.,max_iter=2000)).fit(a,c)
proba=pipe.predict_proba(b)[:,1]
print('ROC AUC:',round(roc_auc_score(d,proba),3))
for t in [.3,.5,.7]:
    precision,recall,f1,_=precision_recall_fscore_support(d,proba>=t,average='binary',zero_division=0)
    print('threshold',t,'precision',round(precision,3),'recall',round(recall,3),'f1',round(f1,3))

ROC AUC: 0.998
threshold 0.3 precision 0.968 recall 1.0 f1 0.984
threshold 0.5 precision 0.989 recall 0.989 f1 0.989
threshold 0.7 precision 0.988 recall 0.933 f1 0.96


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
import numpy as np
coef=pipe.named_steps['logisticregression'].coef_[0]
print('Coeficientes maiores em magnitude (atributos padronizados):',np.argsort(np.abs(coef))[-5:][::-1])
print('Odds ratio ilustrativa para +1 unidade PADRONIZADA no primeiro atributo:',round(float(np.exp(coef[0])),3))

Coeficientes maiores em magnitude (atributos padronizados): [21 10 28 27 13]
Odds ratio ilustrativa para +1 unidade PADRONIZADA no primeiro atributo: 0.593


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: converta log-odds em probabilidade
import numpy as np
log_odds=.8;p=1/(1+np.exp(-log_odds))
print('Probabilidade:',round(float(p),3),'odds:',round(float(p/(1-p)),3))

Probabilidade: 0.69 odds: 2.226


## 6. Interpretação, limitações e erros comuns

Coeficientes não representam causalidade. Tuning de limiar deve usar validação e nunca o teste final. Em contexto de saúde, nomes de classes e custos dos erros precisam ser explicitados antes de decidir o limiar.

**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- `C` é inversamente relacionado à força da regularização.
- Escalonar antes da penalização, dentro do pipeline.
- `roc_auc_score` usa scores/probabilidades, não rótulos binários.
- Coeficientes dependem de escala e colinearidade.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular

**Origem principal das aulas:** 11-logistic-regression.pdf; Disciplina 03 — NB10.

**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** probabilidade calibrada. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **probabilidade calibrada** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **probabilidade calibrada** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
